In [14]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns


In [15]:
# Generate time range (5-minute interval from 1-1-2023 to 31-12-2024)
date_range = pd.date_range(start="2023-01-01", end="2024-12-31 23:55", freq="5min")

# Total entries
total_entries = len(date_range)
print("Total entries:", total_entries)


Total entries: 210528


In [16]:
# Split equally into High, Moderate, Low
categories = ["High", "Moderate", "Low"]
entries_per_category = total_entries // len(categories)

# Assign mold risk categories evenly
mold_risk = np.repeat(categories, entries_per_category)
mold_risk = np.concatenate([mold_risk, np.random.choice(categories, total_entries - len(mold_risk))])

# Shuffle to distribute categories randomly across the timeline
np.random.shuffle(mold_risk)

print(pd.Series(mold_risk).value_counts())


Low         70176
Moderate    70176
High        70176
Name: count, dtype: int64


In [17]:
def generate_conditions(risk):
    # Base temperature
    base_temp = np.random.uniform(15, 30)

    # Make humidity partially dependent on temperature (to increase correlation)
    if risk == "High":
        temperature = np.clip(base_temp + np.random.uniform(2, 6), 18, 35)
        humidity = np.clip(70 + (temperature * 0.8) + np.random.uniform(-5, 5), 75, 100)
        light = np.clip(300 - (humidity * 1.5) + np.random.uniform(-50, 50), 0, 300)
        pressure = np.random.uniform(990, 1010)

    elif risk == "Moderate":
        temperature = np.clip(base_temp + np.random.uniform(-2, 4), 16, 34)
        humidity = np.clip(55 + (temperature * 0.7) + np.random.uniform(-10, 10), 60, 85)
        light = np.clip(400 - (humidity * 1.0) + np.random.uniform(-50, 100), 150, 500)
        pressure = np.random.uniform(1008, 1015)

    else:  # Low
        temperature = np.clip(base_temp + np.random.uniform(-5, 2), 10, 28)
        humidity = np.clip(40 + (temperature * 0.6) + np.random.uniform(-15, 5), 30, 70)
        light = np.clip((humidity * 1.2) + np.random.uniform(200, 400), 200, 1000)
        pressure = np.random.uniform(1012, 1030)

    return humidity, temperature, light, pressure


In [18]:
humidity, temperature, light, pressure = [], [], [], []

for risk in mold_risk:
    h, t, l, p = generate_conditions(risk)
    humidity.append(h)
    temperature.append(t)
    light.append(l)
    pressure.append(p)


In [19]:
# Room types
room_types = ["Living Room", "Bedroom", "Kitchen", "Bathroom"]
room_type = np.random.choice(room_types, total_entries)

# Coordinates
x_coord = np.random.randint(0, 50, total_entries)
y_coord = np.random.randint(0, 50, total_entries)

# Mold growth rate (depends on humidity & risk)
growth_rate = []
for h, risk in zip(humidity, mold_risk):
    if risk == "High":
        growth_rate.append(np.random.uniform(0.7,1.0) * h/100)
    elif risk == "Moderate":
        growth_rate.append(np.random.uniform(0.4,0.7) * h/100)
    else:
        growth_rate.append(np.random.uniform(0.1,0.4) * h/100)


In [20]:
df = pd.DataFrame({
    "date": date_range.date,
    "timestamp": date_range.time,
    "temperature": temperature,
    "humidity": humidity,
    "light": light,
    "pressure": pressure,
    "mold_risk_category": mold_risk,
    "room_type": room_type,
    "x_coordinate": x_coord,
    "y_coordinate": y_coord,
    "mold_growth_rate": growth_rate
})

df.head()


,date,timestamp,temperature,humidity,light,pressure,mold_risk_category,room_type,x_coordinate,y_coordinate,mold_growth_rate
0,2023-01-01,00:00:00,19.532265,43.395923,268.069537,1014.158334,Low,Kitchen,28,24,0.070354
1,2023-01-01,00:05:00,29.792879,77.162943,361.565517,1010.410460,Moderate,Living Room,17,33,0.314482
2,2023-01-01,00:10:00,18.588704,76.771950,290.793023,1009.276652,Moderate,Living Room,8,1,0.335065
3,2023-01-01,00:15:00,21.540002,41.480259,322.553315,1015.095960,Low,Bedroom,40,5,0.163201
4,2023-01-01,00:20:00,25.561456,55.989453,431.299210,1016.551701,Low,Bathroom,4,21,0.125953


In [21]:
# Count distribution of mold risk categories
print(df["mold_risk_category"].value_counts())

# Summary statistics
print(df.describe())


mold_risk_category
Low         70176
Moderate    70176
High        70176
Name: count, dtype: int64
         temperature       humidity          light       pressure  \
count  210528.000000  210528.000000  210528.000000  210528.000000   
mean       23.669727      70.087030     291.275071    1010.824473   
std         5.087739      18.779952     101.472377       9.772939   
min        10.013077      31.376742     100.012634     990.000285   
25%        19.893303      52.554627     187.953458    1004.998347   
50%        23.692692      71.459505     312.116877    1011.479896   
75%        27.598590      87.929317     376.670714    1016.508577   
max        35.000000     100.000000     473.602489    1029.999976   

        x_coordinate   y_coordinate  mold_growth_rate  
count  210528.000000  210528.000000     210528.000000  
mean       24.429981      24.467686          0.429272  
std        14.401402      14.412719          0.278244  
min         0.000000       0.000000          0.032530  

In [22]:
# Save dataset to CSV file
df.to_csv("mold_dataset_balanced.csv", index=False)

print("Dataset saved as mold_risk_dataset.csv")


Dataset saved as mold_risk_dataset.csv


In [23]:
# Calculate min, max, and average for environmental parameters
stats = df[["temperature", "pressure", "light", "humidity"]].agg(
    ["min", "max", "mean"]
).T.rename(columns={"min": "Minimum", "max": "Maximum", "mean": "Average"})

print(stats)


                Minimum      Maximum      Average
temperature   10.013077    35.000000    23.669727
pressure     990.000285  1029.999976  1010.824473
light        100.012634   473.602489   291.275071
humidity      31.376742   100.000000    70.087030
